# Declaration of Originality

**School of Informatics & IT**
<br/>**Diploma in Applied Artificial Intelligence**
<br/>**Machine Learning for Developers (CAI2C08)**
<br/>**AY2026/2027 April Semester**
<br/>**Program Codes**

* Student Name:



**Declaration of Originality**
* I am the originator of this work, and I have appropriately acknowledged all other original sources used as my references for this work.
* I understand that Plagiarism is the act of taking and using the whole or any part of another person’s work, including work generated by AI, and presenting it as my own.
* I understand that Plagiarism is an academic offence and if I am found to have committed or abetted the offence of plagiarism in relation to this submitted work, disciplinary action will be enforced.

# Libraries

In [ ]:
## Import libraries

# Core data handling
import pandas as pd
import numpy as np

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Model training and evaluation
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

# Saving the trained model for the Streamlit app
import joblib

# 1. Business Understanding
Goal: ?

Goal:predict whether a live shopping session will end in a purchase, so the retailer can intervene before the visitor leaves.

An online retailer takes in thousands of sessions a day, and only around 15% of them end in a purchase. The other 85% leave without buying, and the business has no signal that a visitor is slipping away until they're already gone by then the only options are to email them later or hope they come back on their own. If the site can score a visitor's purchase intent while the session is still live, it can act on that score in the moment: surface a live chat prompt, remind them how close they are to free delivery, hold the visitor before they close the tab. This project builds that scoring model a binary classifier that predicts whether a session will end in a purchase (Revenue), using only the behavioural signals the site already measures as the session happens, such as pages visited, time on each page type, and how the visitor arrived. Success is defined by how well the model catches the visitors who are likely to buy, because the two kinds of mistake are not equally costly: a missed buyer is a lost sale, while a visitor wrongly flagged as a buyer costs only an unnecessary prompt the visitor can ignore. That asymmetry is why the model is tuned to protect against missed buyers first.

# 2. Data Understanding

## 2.1 Load dataset

In [ ]:
##Load the dataset
df = pd.read_csv('../data/online_shoppers_intention.csv')
print(df.shape)
df.head()

(12330, 18)


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False,False
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,Feb,2,2,1,2,Returning_Visitor,False,False
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False,False
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,False,False
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,Feb,3,3,1,4,Returning_Visitor,True,False


## 2.2 Summary Statistics

In [ ]:
## Understand the type of variable for each column  
df.info()


OperatingSystems, Browser, Region, and TrafficType are stored as int64, but they're not real quantities: they're ID codes for categories (e.g. TrafficType 8 isn't "twice" TrafficType 4). Fed straight into a linear model, those numbers would imply an ordering and distance between categories that doesn't exist, so the model would learn a fake relationship from the arbitrary label numbering. They'll need one-hot encoding before modeling, the same way Month and VisitorType will.

Administrative, Informational, and ProductRelated, by contrast, are genuine counts (number of pages of that type visited in the session). Doubling the value doubles the meaning, so they stay numeric as-is.

In [7]:
## Check for missing data
print(df.isnull().sum())

Administrative             0
Administrative_Duration    0
Informational              0
Informational_Duration     0
ProductRelated             0
ProductRelated_Duration    0
BounceRates                0
ExitRates                  0
PageValues                 0
SpecialDay                 0
Month                      0
OperatingSystems           0
Browser                    0
Region                     0
TrafficType                0
VisitorType                0
Weekend                    0
Revenue                    0
dtype: int64


All 18 columns report zero nulls, so no imputation or row-dropping is needed, which means no imputation bias to worry about and nothing to justify or defend in preprocessing. That said, the absence of nulls doesn't guarantee the absence of missing data, so I also checked describe() and the value counts for sentinel values disguised as data (-1, 0, 999, ?) and found none, so the zero-null result holds.

In [13]:
## Describe data distribution
df.describe()

,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,OperatingSystems,Browser,Region,TrafficType
count,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000
mean,2.315166,80.818611,0.503569,34.472398,31.731468,1194.746220,0.022191,0.043073,5.889258,0.061427,2.124006,2.357097,3.147364,4.069586
std,3.321784,176.779107,1.270156,140.749294,44.475503,1913.669288,0.048488,0.048597,18.568437,0.198917,0.911325,1.717277,2.401591,4.025169
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000,1.000000
25%,0.000000,0.000000,0.000000,0.000000,7.000000,184.137500,0.000000,0.014286,0.000000,0.000000,2.000000,2.000000,1.000000,2.000000
50%,1.000000,7.500000,0.000000,0.000000,18.000000,598.936905,0.003112,0.025156,0.000000,0.000000,2.000000,2.000000,3.000000,2.000000
75%,4.000000,93.256250,0.000000,0.000000,38.000000,1464.157214,0.016813,0.050000,0.000000,0.000000,3.000000,2.000000,4.000000,4.000000
max,27.000000,3398.750000,24.000000,2549.375000,705.000000,63973.522230,0.200000,0.200000,361.763742,1.000000,8.000000,13.000000,9.000000,20.000000


The numeric distributions are heavily right-skewed: every mean sits above its median, with Administrative_Duration at a median of 7.5 against a mean of 80.8, and ProductRelated_Duration reaching 63,974 seconds against a median of 599. PageValues has both its median and 75th percentile at zero, so at least 75% of sessions carry no page value. The zeros are semantically real rather than missing — 0 for Administrative_Duration means no time spent on administrative pages, not an unknown value — and no -1 or 999 sentinels appear anywhere, confirming the earlier null check.

Two separate implications for modelling. Scale: features range from bounded rates in [0, 0.2] to durations in the tens of thousands, and because L2 regularisation penalises squared coefficients without regard to units, a large-scale feature escapes the penalty almost entirely while a bounded one is shrunk hard — so LogisticRegression needs StandardScaler, fitted on the training set only. Skew: standardisation rescales but doesn't reshape, so extreme sessions retain disproportionate leverage; a log transform of the duration columns is an option if the linear model underperforms. Tree-based models are unaffected by either, since splits depend on ordering rather than magnitude.

## 2.3 Data Visualization

### 2.3.1 Understanding distribution of data

### 2.3.1.1 Understanding distribution of target

In [ ]:
## Understanding distribution of target
sns.countplot(x='Revenue', data=df)
plt.title('Distribution of Revenue (purchase vs no purchase)')
plt.xlabel('Purchase made')
plt.ylabel('Number of sessions')
plt.show()

print(df['Revenue'].value_counts(normalize=True))

### 2.3.1.2 Understanding distribution of features

In [ ]:
## Understanding distribution of features
numeric_cols = ['Administrative', 'Administrative_Duration', 'Informational',
                'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration',
                'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay']

df[numeric_cols].hist(figsize=(15, 10), bins=30)
plt.tight_layout()
plt.show()

### 2.3.2 Understanding relationship between variables

In [ ]:
## Understanding relationship between variables
plt.figure(figsize=(10, 8))
sns.heatmap(df[numeric_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation between numeric features')
plt.show()

In [ ]:
## PageValues by purchase outcome
df.groupby('Revenue')['PageValues'].describe()

# 3. Data Preparation

## 3.1 Data Cleaning

In [ ]:
## Clean data

## 3.2 Train-Test Split

In [ ]:
## Split data into train set and test set


# 4. Modelling

### 4.2 Train Model

In [ ]:
## Initialise and train model


# 5. Model Evaluation

In [ ]:
## Evaluate model


In [ ]:
## New data

## Predict


## Iterative model development


In [ ]:
## Further feature engineering / feature selection